# Virtual Embryo data tour for ML people

This notebook is a **data/mental-model tour**, not a competition baseline.

It uses only the tiny example files that the organizers publish with [`veckit`](https://github.com/aristoteleo/veckit). No challenge login or hidden labels are needed.

By the end, you should be able to answer:

1. What does one row of an expression matrix represent?
2. Why is a developmental stage a *distribution of cells* rather than a normal supervised target?
3. What extra object appears in the spatial tasks?
4. Why can wild type look extremely similar to a knockout even when the perturbation response matters?

The mini files are intentionally small. They are good for inspection and plotting, not for drawing biological conclusions.

In [ ]:
# Install the organizers' public scorer, which also installs AnnData and the dependencies we use.
# Pinning a known public commit keeps this notebook reproducible even if the default branch changes.
!pip -q install "git+https://github.com/aristoteleo/veckit.git@46d41e63f42a9aab815db20b742feeccd249cb17" matplotlib pandas

In [ ]:
from pathlib import Path
import urllib.request

DATA = Path('/content/virtual_embryo_mini')
DATA.mkdir(exist_ok=True)

base = 'https://raw.githubusercontent.com/aristoteleo/veckit/46d41e63f42a9aab815db20b742feeccd249cb17/data/'
files = [
    'sample_8.5.h5ad',
    'sample_9.5.h5ad',
    'sample_heart_9.25.h5ad',
    'sample_heart_9.5.h5ad',
    'sample_wt.h5ad',
    'sample_mab21l2_ko.h5ad',
]

for name in files:
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(base + name, path)
    print(f'{name:28s} {path.stat().st_size / 1e6:6.2f} MB')

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.decomposition import PCA


def dense_X(a):
    return a.X.toarray().astype(np.float32) if sparse.issparse(a.X) else np.asarray(a.X, dtype=np.float32)


def describe(a, name):
    print(name)
    print('  cells × genes:', a.shape)
    print('  obs columns:', list(a.obs.columns))
    print('  has spatial_3D:', 'spatial_3D' in a.obsm)
    print('  first genes:', list(a.var_names[:5]))
    x = dense_X(a)
    print('  expression range:', float(x.min()), 'to', float(x.max()))
    print()


## 1. Task 1: a stage is a population, not one example

Load two public single-cell RNA examples. Each row is one measured cell. Each column is one gene.

In [ ]:
e85 = ad.read_h5ad(DATA / 'sample_8.5.h5ad')
e95 = ad.read_h5ad(DATA / 'sample_9.5.h5ad')

describe(e85, 'E8.5 mini example')
describe(e95, 'E9.5 mini example')

The important consequence is that there is no natural target pair

`E8.5 cell i -> E9.5 cell i`.

The experiments measure different cells, and development includes division and differentiation. What we can compare are the **populations**.

In [ ]:
# If training cell-type labels are present, compare how the mixture changes between stages.
def proportions(a):
    if 'celltype' not in a.obs:
        return None
    return a.obs['celltype'].astype(str).value_counts(normalize=True)

p85 = proportions(e85)
p95 = proportions(e95)

if p85 is not None and p95 is not None:
    props = pd.concat([p85.rename('E8.5'), p95.rename('E9.5')], axis=1).fillna(0)
    props['max_share'] = props.max(axis=1)
    display(props.sort_values('max_share', ascending=False).head(15).drop(columns='max_share').round(3))
else:
    print('These mini files do not expose celltype labels.')

In [ ]:
# Compare average expression (pseudobulk) across stages.
X85 = dense_X(e85)
X95 = dense_X(e95)
pb85 = X85.mean(axis=0)
pb95 = X95.mean(axis=0)
delta = pb95 - pb85

print('pseudobulk correlation:', np.corrcoef(pb85, pb95)[0, 1])
print('\nGenes with largest average change in this mini sample:')
for j in np.argsort(np.abs(delta))[-10:][::-1]:
    print(f'{e85.var_names[j]:>16s}  change={delta[j]: .4f}')

A high pseudobulk correlation does **not** mean the two cell populations are the same. The mean can stay similar while mixture proportions and within-state structure change.

In [ ]:
# A 2D PCA is only a visualization, but it makes the "two clouds of cells" idea concrete.
# We use the most variable genes in this tiny sample to keep the plot light.
X = np.vstack([X85, X95])
stage = np.array(['E8.5'] * len(X85) + ['E9.5'] * len(X95))

var = X.var(axis=0)
keep = np.argsort(var)[-min(1000, X.shape[1]):]
Z = PCA(n_components=2, random_state=0).fit_transform(X[:, keep])

plt.figure(figsize=(7, 5))
for s in ['E8.5', 'E9.5']:
    m = stage == s
    plt.scatter(Z[m, 0], Z[m, 1], s=18, alpha=0.7, label=s)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Two developmental stages are two cell populations')
plt.legend()
plt.show()

## 2. Task 2: cells now have positions

The spatial tasks add `obsm["spatial_3D"]`. Each cell is now a gene-expression vector **and** a point in 3D.

Do not interpret the absolute xyz axes as a global atlas shared across stages. The challenge uses per-embryo local frames, and the scorer is designed around relative geometry.

In [ ]:
h925 = ad.read_h5ad(DATA / 'sample_heart_9.25.h5ad')
h95 = ad.read_h5ad(DATA / 'sample_heart_9.5.h5ad')

describe(h925, 'Heart E9.25 mini example')
describe(h95, 'Heart E9.5 mini example')

for name, a in [('E9.25', h925), ('E9.5', h95)]:
    C3 = np.asarray(a.obsm['spatial_3D'])[:, :3]
    plt.figure(figsize=(6, 5))
    plt.scatter(C3[:, 0], C3[:, 1], s=15, alpha=0.75)
    plt.xlabel('local x')
    plt.ylabel('local y')
    plt.title(f'{name}: one projection of the spatial point cloud')
    plt.axis('equal')
    plt.show()

A useful mental model for Task 2 is a **point cloud with biological features attached to each point**.

That means a model can fail in several independent ways:

- realistic expression, wrong global shape
- plausible shape, wrong cell-state mixture
- both marginals look good, but the wrong cell states are neighbors

This is why the task scores molecular, cellular, global spatial, and local spatial questions separately.

## 3. Task 3: the response is much smaller than the whole embryo

Load the public matched-WT and Mab21l2 knockout mini examples. We will compare the *absolute state* with the *WT -> KO change*.

In [ ]:
wt = ad.read_h5ad(DATA / 'sample_wt.h5ad')
ko = ad.read_h5ad(DATA / 'sample_mab21l2_ko.h5ad')

describe(wt, 'Matched WT mini example')
describe(ko, 'Mab21l2 KO mini example')

Xwt = dense_X(wt)
Xko = dense_X(ko)
pb_wt = Xwt.mean(axis=0)
pb_ko = Xko.mean(axis=0)
response = pb_ko - pb_wt

print('absolute WT/KO pseudobulk correlation:', np.corrcoef(pb_wt, pb_ko)[0, 1])
print('mean absolute response:', np.abs(response).mean())
print('\nLargest response genes in this mini sample:')
for j in np.argsort(np.abs(response))[-10:][::-1]:
    print(f'{wt.var_names[j]:>16s}  WT->KO={response[j]: .4f}')

The absolute correlation can be very high because most of the embryo is still similar to wild type.

That is why Task 3 is easier to reason about in terms of the perturbation response

\[
\Delta = \bar x_{KO} - \bar x_{WT}.
\]

A model that returns WT unchanged can look superficially plausible while predicting \(\Delta = 0\), which is exactly the biologically interesting part it was supposed to model.

In [ ]:
# Make three deliberately simple response vectors just to see the distinction.
responses = {
    'no response': np.zeros_like(response),
    'known mean response': response,
    'reversed response': -response,
}

for name, r in responses.items():
    direction_cos = float(np.dot(r, response) / (np.linalg.norm(r) * np.linalg.norm(response) + 1e-12))
    magnitude_ratio = float(np.linalg.norm(r) / (np.linalg.norm(response) + 1e-12))
    print(f'{name:22s} direction cosine={direction_cos: .3f}  magnitude ratio={magnitude_ratio: .3f}')

For the official response metrics and controlled score experiments, continue with the **Task 3 Response Playground** in this repository. That notebook intentionally makes the response too weak, too strong, reversed, and shuffled across genes, then runs the official scorer.

## 4. What should you model first?

A useful progression is:

1. **Copy last / WT identity** — prove the pipeline works.
2. **Global mean shift** — can you at least move average expression in the right direction?
3. **State-aware shifts + state proportions** — different cell populations develop differently.
4. **Soft distribution matching / optimal transport** — stop pretending cells are paired across stages.
5. **Latent developmental dynamics** — learn a continuous time-conditioned generator.
6. **Perturbation conditioning** — for Task 3, transfer response structure across genes.

The right next model is the smallest one that can represent the failure your current model cannot.

## 5. Before you leave this notebook

Make sure these statements feel obvious:

- A stage is a **set/distribution of cells**, not one feature vector.
- The challenge does not assume cell-to-cell correspondence across stages.
- Pseudobulk is useful but throws away multimodal population structure.
- Task 2 adds a 3D tissue-organization problem, not just three extra regression columns.
- Task 3 is about the **change caused by a knockout**, not merely looking like a mutant embryo in absolute expression.

Then read [`guide.md`](../guide.md) for the modeling ladder and task-by-task failure modes.

### Sources

- https://virtualembryo.ai/challenge/tasks
- https://virtualembryo.ai/challenge/evaluation
- https://github.com/aristoteleo/veckit

Independent community notebook. The official challenge site remains the source of truth.